# 🧠 LIVR-Mini-Benchmark: GIAI ĐOẠN HUẤN LUYỆN (IMPLEMENT MINI)
Notebook này thực hiện huấn luyện mô hình **LIVR (Latent Implicit Visual Reasoning)** trên kiến trúc **Qwen2.5-VL-3B-Instruct**.

### Quy trình kỹ thuật tổng quát:
1. **Git Setup & Setup Environment**: Đồng bộ mã nguồn từ GitHub cá nhân (nhánh `develop`) và cài đặt các thư viện lõi.
2. **Dữ liệu**: Nạp tập dữ liệu huấn luyện `LIVR_mixed` và chạy pipeline tiền xử lý, bao gồm lọc dải đối tượng đếm và khử trùng lặp ảnh (Visual De-duplication).
3. **Kiến trúc mô hình**: Khởi tạo Qwen2.5-VL, chèn K=16 Latent Tokens, thiết lập LoRA và áp dụng Hook để đóng băng toàn bộ bảng nhúng gốc ngoại trừ Latent Tokens.
4. **Huấn luyện**: Chạy huấn luyện 2 giai đoạn: **Stage 1 (Bottleneck Mask - 2 epochs)** và **Stage 2 (Unmasked Causal Mask - 3 epochs)** với kỹ thuật **Tích lũy Gradient (Gradient Accumulation)**.
5. **Trực quan hóa**: Vẽ biểu đồ Loss của 2 giai đoạn để chứng minh sự hội tụ.
6. **Nghiệm thu đối chiếu**: Đánh giá Accuracy so sánh giữa mô hình LIVR và mô hình gốc của hãng.

## 🛠️ 1. Đồng Bộ Mã Nguồn & Thiết Lập Google Colab
Phần này thực hiện kết nối với Google Drive để chuẩn bị lưu trữ checkpoints và tải mã nguồn từ nhánh `develop` của bạn.
*   **Mục tiêu**: Giúp luồng phát triển code độc lập trên máy local đồng bộ mượt mà với môi trường tính toán mạnh mẽ của Colab qua GitHub.
*   **Luồng hoạt động**: Sử dụng lệnh hệ thống để kiểm tra thư mục hiện tại, clone repo mới nếu chưa có, hoặc chạy `git pull` để nhận các cập nhật mới nhất từ nhánh `develop`.

In [ ]:
# =========================================================================
# CELL 1: KẾT NỐI GOOGLE DRIVE & ĐỒNG BỘ CODE TỪ GITHUB (DEVELOP BRANCH)
# =========================================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Cấu hình URL repository của bạn
REPO_URL = "https://github.com/YOUR_USERNAME/LIVR-Mini-Benchmark.git"
PROJECT_DIR = "LIVR-Mini-Benchmark"
BRANCH = "develop"

%cd /content
import os
if not os.path.exists(PROJECT_DIR):
    print(f"---> Đang thực hiện clone repo {REPO_URL} (nhánh {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL}
    %cd {PROJECT_DIR}
else:
    print(f"---> Repo {PROJECT_DIR} đã tồn tại. Đang tiến hành pull code mới nhất từ nhánh {BRANCH}...")
    %cd {PROJECT_DIR}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

## 📊 2. Tiền Xử Lý Dữ Liệu & Lọc Trùng Lặp Ảnh Trực Quan (Visual De-duplication)
Phần này cài đặt các dependencies và chạy pipeline xử lý dữ liệu thô từ Hugging Face.
### Các khái niệm & Thuật ngữ chính:
1. **Perceptual Hashing (Mã băm nhận thức - pHash)**:
   * Khác với mã băm mật mã (như MD5/SHA256 - chỉ cần đổi 1 pixel là mã băm đổi hoàn toàn), **pHash** biểu diễn các đặc trưng cấu trúc tần số thấp của ảnh.
   * Hai ảnh tương đồng trực quan (ví dụ: cùng góc chụp nhưng lệch sáng, hoặc bị nén nhẹ) sẽ có mã băm pHash rất gần nhau (khoảng cách Hamming giữa chúng nhỏ).
   * Công thức Khoảng cách Hamming:
     $$D_H(x, y) = \sum_{i=1}^{d} (x_i \neq y_i)$$
     Nếu $D_H \le 5$, chúng ta coi như hai bức ảnh bị trùng lặp cấu trúc thị giác và loại bỏ khỏi tập Train để tránh rò rỉ tri thức (Data Leakage) sang tập Test.
2. **Dải đếm (Counting Range)**:
   * Theo tài liệu của tác giả bài báo LIVR, chỉ giữ lại các đối tượng đếm nằm trong khoảng $[2, 10]$ để tránh phân phối dữ liệu bị lệch và tối ưu hóa khả năng nhận thức của mô hình.

In [ ]:
# =========================================================================
# CELL 2: CÀI ĐẶT THƯ VIỆN & CHẠY PIPELINE TIỀN XỬ LÝ DỮ LIỆU (TUẦN 1 & TUẦN 2)
# =========================================================================
# Cài đặt các thư viện cần thiết trên môi trường Colab
!pip install -r requirements.txt

import sys
import os
# Đảm bảo Python nhận diện được module trong thư mục src/
sys.path.append(os.getcwd())

# 1. Nạp và kiểm tra dữ liệu gốc (Nghiệm thu Tuần 1)
from src.utils import load_and_inspect_livr_dataset, filter_and_deduplicate_pipeline
dataset = load_and_inspect_livr_dataset()

# 2. Chạy thử nghiệm bộ tiền xử lý và khử trùng lặp trực quan (Nghiệm thu Tuần 2)
cleaned_dataset = filter_and_deduplicate_pipeline(dataset)

## 🛠️ 3. Cấu Hình Kiến Trúc LIVR & Tham Số Hóa Thấp (PEFT LoRA)
Phần này khởi tạo mô hình nền tảng, nới rộng bảng từ vựng cho Latent Tokens và đóng băng có chọn lọc các tham số.
### Các khái niệm & Công thức kỹ thuật:
1. **Mở rộng bảng từ vựng (Vocab Expansion)**:
   * Chúng ta chèn thêm $K=16$ token đặc biệt (`<latent_0>` đến `<latent_15>`) vào bảng từ vựng của tokenizer và nới rộng tầng embedding của mô hình.
2. **PEFT LoRA (Low-Rank Adaptation)**:
   * Giúp huấn luyện mô hình lớn với tài nguyên cực nhỏ bằng cách đóng băng trọng số gốc $W_0 \in \mathbb{R}^{d \times k}$ và bổ sung 2 ma trận phân rã hạng thấp $A \in \mathbb{R}^{r \times k}$ và $B \in \mathbb{R}^{d \times r}$ (với hạng $r \ll \min(d, k)$, mặc định $r=16$).
   * Công thức lan truyền xuôi của trọng số LoRA:
     $$h = W_0 x + \Delta W x = W_0 x + \frac{\alpha}{r} (B \cdot A) x$$
     Trong đó $\alpha$ là hằng số tỉ lệ (scaling hyperparameter).
3. **Embedding Backward Hook (Khóa cứng biểu diễn cũ)**:
   * Chúng ta unfreeze ma trận embedding để cập nhật vector cho 16 Latent Tokens ngẫu nhiên ban đầu. Tuy nhiên, để ngăn các từ gốc bị lệch nghĩa (Token Drift), ta đăng ký một **Backward Hook** trên ma trận gradient của bảng nhúng.
   * Hook này nhân ma trận gradient $\nabla E$ với mặt nạ nhị phân $M \in \{0, 1\}^{V \times 1}$:
     $$\nabla E_{\text{hooked}} = \nabla E \odot M$$
     Trong đó $M_i = 1$ nếu $i$ thuộc danh mục index của Latent Tokens, ngược lại $M_i = 0$. Điều này triệt tiêu hoàn toàn gradient của các từ gốc và chỉ cập nhật các Latent Tokens.

In [ ]:
# =========================================================================
# CELL 3: ĐỌC CONFIG, KHỞI TẠO MÔ HÌNH VÀ BỘ TỐI ƯU (TUẦN 4 - BƯỚC 1)
# =========================================================================
import json
import torch
from transformers import AdamW
from src.model import LIVRModelManager
from src.mask import patch_model_for_livr

# 1. Đọc file cấu hình định nghĩa sẵn
with open("config/implement_config.json", "r", encoding="utf-8") as f:
    config = json.load(f)
print("IMPLEMENTATION CONFIGURATION:")
print(json.dumps(config, indent=2))

# 2. Dựng mô hình LIVR cấu hình K từ file config
manager = LIVRModelManager(model_id=config["model_id"], K=config["K"], device="cuda")
model = manager.setup_peft_and_freezing(
    r=config["lora_r"],
    alpha=config["lora_alpha"],
    dropout=config["lora_dropout"]
)
processor = manager.processor

# Monkey-patch model với Custom Attention Mask
patch_model_for_livr(
    model=model,
    latent_token_ids=manager.latent_token_ids,
    image_pad_token_id=manager.image_pad_token_id,
    pad_token_id=manager.pad_token_id
)

# 3. Khai báo bộ tối ưu AdamW sử dụng cấu hình từ file config
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=config["stage1_lr"], weight_decay=config["weight_decay"])

print("-> Đã khởi tạo cấu hình model và Optimizer thành công.")

## 🔄 4. Vòng Lặp Huấn Luyện 2 Giai Đoạn & Tích Lũy Gradient
Phần này chạy quá trình huấn luyện chính thức trên dữ liệu sạch.
### Luồng hoạt động & Kỹ thuật cốt lõi:
1. **Stage 1 (Bottleneck Phase - 2 Epochs đầu)**:
   * Áp dụng ma trận Attention Mask chặn đứng tương tác trực tiếp giữa Prompt/Answer tokens và Image tokens.
   * Ma trận chú ý tùy biến có dạng:
     $$\text{mask}[\text{Answer}, \text{Image}] = -\infty$$
     $$\text{mask}[\text{Prompt}, \text{Image}] = -\infty$$
   * Mọi luồng thông tin ảnh buộc phải nén và truyền tải qua **Latent Tokens** trước khi nuôi câu trả lời. Điều này thúc đẩy tạo biểu diễn ngầm ẩn (Forced Implicit Representation).
2. **Stage 2 (Unmasked Phase - 3 Epochs sau)**:
   * Trả lại ma trận Causal Attention tiêu chuẩn của hệ thống Qwen.
   * Câu trả lời được tiếp cận cả đặc trưng ảnh chi tiết mức thấp (raw pixels) và tri thức cấu trúc mức cao đã học được trong Latent Tokens từ Stage 1.
3. **Tích lũy Gradient (Gradient Accumulation)**:
   * Để giả lập kích thước batch hiệu dụng lớn (Effective Batch Size = 8) trên GPU VRAM thấp (như 15GB T4/L4), chúng ta chỉ chạy Batch kích thước thực tế là 1, và cộng dồn gradient qua $N=8$ bước trước khi cập nhật trọng số:
     $$\Delta W \leftarrow \Delta W + \frac{1}{N} \sum_{i=1}^{N} \nabla \mathcal{L}_i$$
     Sau đúng 8 bước, ta chạy `optimizer.step()` và xóa gradient nháp bằng `optimizer.zero_grad()`.

In [ ]:
# =========================================================================
# CELL 4: VÒNG LẶP HUẤN LUYỆN CHÍNH (TRAINING LOOP - TUẦN 4 - BƯỚC 2)
# =========================================================================
import os
from tqdm import tqdm
from src.utils import prepare_vqa_inputs

# Lấy các tham số huấn luyện trực tiếp từ file config
STAGE1_EPOCHS = config["stage1_epochs"]
STAGE2_EPOCHS = config["stage2_epochs"]
TOTAL_EPOCHS = STAGE1_EPOCHS + STAGE2_EPOCHS
GRADIENT_ACCUMULATION_STEPS = config["grad_accumulation_steps"]
STAGE2_LR = config["stage2_lr"]
drive_checkpoint_dir = config["output_dir"]

model.train()
epoch_losses = []

print("====== CHÍNH THỨC KHỞI ĐỘNG VÒNG LẶP HUẤN LUYỆN 2 GIAI ĐOẠN ======")

for epoch in range(1, TOTAL_EPOCHS + 1):
    # Xác định Giai đoạn hiện tại để điều khiển mặt nạ mã nguồn
    current_stage = 1 if epoch <= STAGE1_EPOCHS else 2
    model.livr_stage = current_stage
    
    # Reset hoặc giảm Learning Rate khi chuyển sang Stage 2 theo đúng paper
    if epoch == STAGE1_EPOCHS + 1:
        print(f"\n➔ CHUYỂN GIAI ĐOẠN: Hạ Learning Rate xuống {STAGE2_LR} cho Stage 2...")
        for param_group in optimizer.param_groups:
            param_group['lr'] = STAGE2_LR
            
    # Khởi tạo loss và reset gradients cho mỗi epoch ở cấp độ vòng lặp epoch
    epoch_loss = 0.0
    optimizer.zero_grad()
    
    # Thanh tiến trình theo dõi tiến độ từng epoch
    progress_bar = tqdm(cleaned_dataset, desc=f"Epoch {epoch}/{TOTAL_EPOCHS} (Stage {current_stage})")
    
    for step, batch in enumerate(progress_bar):
        # Sử dụng hàm prepare_vqa_inputs để chuẩn bị Tensor đầu vào đạt chuẩn cho Qwen
        inputs = prepare_vqa_inputs(
            processor=processor,
            conversation=batch['conversation'],
            latent_tokens=manager.latent_tokens,
            device="cuda"
        )
        
        # --- KỸ THUẬT CAN THIỆP PHÂN PHỐI MẶT NẠ CHÚ Ý (STAGING) ---
        # Hàm forward của model đã được chúng ta đè (Monkey-patched) từ tuần trước.
        # Hệ thống tự động sử dụng model.livr_stage để sinh ra mask tương ứng.
        outputs = model(**inputs)
        
        # Lấy giá trị Loss của lượt chạy này và chia cho các bước tích lũy
        loss = outputs.loss / GRADIENT_ACCUMULATION_STEPS
        loss.backward() # Lan truyền ngược ghi đạo hàm vào giấy nháp
        
        epoch_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS
        
        # --- THỰC THI TÍCH LŨY GRADIENT CHỐNG TRÀN VRAM CHUẨN Ý THẦY ---
        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0 or (step + 1) == len(cleaned_dataset):
            # Cắt bớt gradient nếu quá lớn để chống bùng nổ đạo hàm (Gradient Clipping)
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            
            optimizer.step()      # Cập nhật bộ não dựa trên giấy nháp tổng hợp từ 8 ảnh
            optimizer.zero_grad()  # Xóa sạch giấy nháp để chuẩn bị cho chu kỳ 8 ảnh tiếp theo
            
        # Cập nhật thông số Loss liên tục lên màn hình console
        progress_bar.set_postfix({"Loss": f"{loss.item() * GRADIENT_ACCUMULATION_STEPS:.4f}"})
        
    avg_loss = epoch_loss / len(cleaned_dataset)
    epoch_losses.append(avg_loss)
    print(f"➔ Kết thúc Epoch {epoch} - Average Loss tổng thể: {avg_loss:.4f}")

    # --- LƯU CHECKPOINT TRUNG GIAN GIAI ĐOẠN 1 (STAGE 1 CHECKPOINT) ---
    if epoch == STAGE1_EPOCHS:
        stage1_checkpoint_path = os.path.join(drive_checkpoint_dir, "livr_stage1_checkpoint.pt")
        os.makedirs(drive_checkpoint_dir, exist_ok=True)
        torch.save({
            'model_state_dict': {k: v.cpu() for k, v in model.state_dict().items() if v.requires_grad},
            'latent_embeddings': model.base_model.model.model.embed_tokens.weight[manager.latent_token_ids].detach().cpu(),
            'latent_token_ids': manager.latent_token_ids
        }, stage1_checkpoint_path)
        print(f"---> Đã lưu checkpoint trung gian Giai đoạn 1 tại: {stage1_checkpoint_path}")

# --- BƯỚC CUỐI: ĐÓNG GÓI VÀ LƯU TRỰC TIẾP CHECKPOINT RA GOOGLE DRIVE ---
os.makedirs(drive_checkpoint_dir, exist_ok=True)
checkpoint_path = os.path.join(drive_checkpoint_dir, "livr_mini_checkpoint.pt")

# Lưu các tham số LoRA weights và Latent Embeddings (các tham số requires_grad=True)
torch.save({
    'model_state_dict': {k: v.cpu() for k, v in model.state_dict().items() if v.requires_grad},
    'latent_embeddings': model.base_model.model.model.embed_tokens.weight[manager.latent_token_ids].detach().cpu(),
    'latent_token_ids': manager.latent_token_ids
}, checkpoint_path)

print(f"\n[SUCCESS] Hoàn thành trọn vẹn phần Implement (Mini)!")
print(f"File trọng số thông minh đã được lưu an toàn tại: {checkpoint_path}")

## 📈 5. Trực Quan Hóa Biểu Đồ Loss Nghiệm Thu
Phần này vẽ đồ thị theo dõi sự sụt giảm của hàm Loss tích lũy qua các Epoch.
*   **Mục tiêu**: Báo cáo tiến độ nghiệm thu kỹ thuật và trực quan hóa hành vi học của mô hình.
*   **Đặc điểm hình thái**: Đường Loss thường giảm mạnh ở đầu Stage 1, sau đó khi chuyển sang Stage 2 (Epoch 3) có thể dao động nhẹ do cấu trúc Attention thay đổi, nhưng sẽ nhanh chóng dốc xuống và hội tụ mượt mà ở các epoch cuối.

In [ ]:
# =========================================================================
# CELL 5: TRỰC QUAN HÓA BIỂU ĐỒ LOSS TRONG 2 GIAI ĐOẠN
# =========================================================================
import matplotlib.pyplot as plt

# Vẽ đồ thị sụt giảm của Loss để nghiệm thu biểu đồ log loss
plt.figure(figsize=(10, 5))
plt.plot(range(1, TOTAL_EPOCHS + 1), epoch_losses, marker='o', color='b', linewidth=2, label='Training Loss')
plt.axvline(x=STAGE1_EPOCHS + 0.5, color='r', linestyle='--', linewidth=1.5, label='Chuyển sang Stage 2 (Unmasked)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Biểu đồ Log Loss sụt giảm qua 2 giai đoạn (LIVR Training)')
plt.xticks(range(1, TOTAL_EPOCHS + 1))
plt.legend()
plt.grid(True)
plt.show()

## 📊 6. Đánh Giá Đối Chiếu Hiệu Năng (Baseline Accuracy Comparison)
Phần này chạy thử nghiệm nghiệm thu đối chiếu hiệu năng để hoàn thành báo cáo kỹ thuật.
*   **Top-1 Accuracy**: Đo lường tỉ lệ phần trăm câu trả lời sinh ra từ mô hình trùng khớp chính xác 100% với nhãn mặt đất (ground-truth targets).
*   **Adapter Disabling**: Chúng ta sử dụng hàm context manager `model.disable_adapter()` để tạm thời ngắt ma trận trọng số LoRA thích ứng, đưa mô hình quay về trạng thái gốc của hãng (Base Model) nhằm thực hiện đánh giá công bằng trên cùng tập mẫu.

In [ ]:
# =========================================================================
# CELL 6: ĐÁNH GIÁ CHỈ SỐ ACCURACY ĐỐI CHIẾU (LIVR VS BASE MODEL)
# =========================================================================
import random
from src.utils import prepare_vqa_inputs

def evaluate_accuracy(model, eval_dataset, max_samples=50):
    model.eval()
    correct = 0
    total = 0
    
    # Lấy ngẫu nhiên mẫu từ cleaned_dataset để chạy đánh giá nhanh đối chiếu
    samples = random.sample(list(eval_dataset), min(max_samples, len(eval_dataset)))
    
    with torch.no_grad():
        for batch in tqdm(samples, desc="Evaluating"):
            inputs = prepare_vqa_inputs(
                processor=processor,
                conversation=batch['conversation'],
                latent_tokens=manager.latent_tokens,
                device="cuda"
            )
            
            # Xóa nhãn labels để chuyển processor sang chế độ Inference
            inputs.pop("labels", None)
            outputs = model.generate(**inputs, max_new_tokens=10)
            
            pred_text = processor.decode(outputs[0], skip_special_tokens=True).strip()
            target_text = str(batch['conversation'][1]['content'][0]['text']).strip()
            
            if pred_text.lower() == target_text.lower():
                correct += 1
            total += 1
            
    return (correct / total) * 100 if total > 0 else 0.0

# 1. Đánh giá mô hình LIVR sau huấn luyện (LoRA bật)
print("---> Đang đánh giá hiệu năng mô hình LIVR...")
model.livr_stage = 2
livr_accuracy = evaluate_accuracy(model, cleaned_dataset)

# 2. Tắt tạm thời LoRA để đánh giá mô hình Gốc (Base Qwen2.5-VL-3B-Instruct)
print("\n---> Đang đánh giá hiệu năng mô hình gốc Base Model...")
with model.disable_adapter():
    base_accuracy = evaluate_accuracy(model, cleaned_dataset)

# 3. Trực quan bảng so sánh hiệu năng để báo cáo thầy
print("\n=========================================================")
print("   BẢNG SO SÁNH HIỆU NĂNG ĐỐI CHIẾU (LIVR VS BASE MODEL)  ")
print("=========================================================")
print(f"| Model | Accuracy (%) |")
print(f"|---|---|")
print(f"| Base Model (Qwen2.5-VL-3B) | {base_accuracy:.2f}% |")
print(f"| LIVR (Mini) | {livr_accuracy:.2f}% |")
print("=========================================================")